# Problem 045:  Triangular, Pentagonal, and Hexagonal

> Triangle, pentagonal, and hexagonal numbers are generated by the following formulae:
>
> |   |   |   |
> |---|---|---|
> | Triangle   | $T_n=n(n+1)/2$    | $1, 3, 6, 10, 15, \dots$  |
> | Pentagonal | $P_n=n(3n - 1)/2$ | $1, 5, 12, 22, 35, \dots$ |
> | Hexagonal  | $H_n=n(2n - 1)$   | $1, 6, 15, 28, 45, \dots$ |
>
> It can be verified that $T_{285} = P_{165} = H_{143} = 40755$.
>
> Find the next triangle number that is also pentagonal and hexagonal.

## First pass Solution $\mathcal O \left( e^N \right)$

It's a race!  A trivial way to do the problem is to just calculate the three values starting at $1$ for all of them, and then increment the smallest value in the list.  A counter is incremented each time the three values are all equal.

If we are looking for the $N^\mbox{th}$ time they are all equal, this algorithm runs in $\mathcal O(e^N)$ time, due to the distribution of solutions as discussed below.

In [57]:
%%time

def p045_slow(tot: int) -> int:
    if tot == 1:
        return 1

    vals = [(1, 1, 0), (1, 1, 1), (1, 1, 2)]
    cnt = 1
    a = [1,3,4]
    while True:
        x, n, i = min(vals)
        x += a[i] * n + 1
        n += 1
        vals[i] = (x, n, i)
        
        if vals[0][0] == vals[1][0] == vals[2][0]:
            cnt += 1
            if cnt == tot:
                return vals[0][0]    
    return 0

N = 3
ans = p045_slow(N)

print(ans)

1533776805
CPU times: user 76.6 ms, sys: 0 ns, total: 76.6 ms
Wall time: 74.8 ms


## Faster Solution $\mathcal O \left( N \right)$

First thing that needs to be pointed out is that all hexagonal numbers are triangle numbers.  There is no need to check that a number is a triangle number if you know it is hexagonal.  That would drop about half the calculations in the above algorithm.

But there is a direct method of calculating the answer.  We are searching for when $P_m = H_n$.  Writing out the equation and solving the quadratic for $n$ gives
$$m (3m - 1) / 2 = n (2n -1) \implies n = \frac{1}{4}\left[ 1 + \sqrt{12m^2 - 4m + 1} \right].$$

The minus sign option in the quadratic equation is ignored because it will always give a negative $n$.  For a valid answer, the radical must be a perfect square.  Call that value $a$.  Do the process of writing out the equation for $a$ and solve the quadratic again:
$$a^2 = 12m^2 - 4m + 1 \implies m = \frac{1}{6}\left[ 1 + \sqrt{3a^2 - 2} \right].$$

The same logic is used to ignore the minus sign option in the quadratic equation.  And, again, the radical must be a perfect square.  Call that value $b$.  Collecting everything gives the following relations:
$$\begin{align}
b^2 - 3a^2 = -2\\
n = \frac{1}{4} (1 + a)\\
m = \frac{1}{6} (1 + b)
\end{align}$$

This first equation is a generalized Pell's equation (Wikipedia: [Pell's equation](https://en.wikipedia.org/wiki/Pell%27s_equation)).  To solve, the solutions to $u^2 - 3v^2 = 1$ are first found recursively, then the fundemental solution(s) to $b^2-3a^2=-2$ are found to generate all the values.

The fundemental solution to $u^2 - 3v^2 = 1$ is $(u_0, v_0) = (2,1)$.  All other solutions are obtained by:
$$\begin{align}
u_{k+1} & = & u_0 u_k + 3 v_0 v_k & = & 2 u_k + 3 v_k\\
v_{k+1} & = & u_0 v_k + u_k v_0 & = & 2 v_k + v_0
\end{align}$$

The only fundemental solution to $b^2 - 3a^2 = -2$ is $(a_0, b_0) = (1,1)$.  All other solutions are obtained by:
$$\begin{align}
b_{k+1} & = & b_0 u_k + 3 a_0 v_k & = & u_k + 3v_k \\
a_{k+1} & = & b_0 v_k + a_0 u_k   & = & u_k + v_k\\
\end{align}$$

This last system of equations give valid values $a$ and $b$ that solve the generalized Pell's eqaution, but also need to generate integers for $n$ and $m$, the index of the hexagonal and pentagonal numbers.  For this, we need $a \% 4 = 3$ and $b \% 6 = 5$.  Instead of deriving the pattern in the recursive equations, I just empiracally observed it by calculating different values of $a_k$ and $b_k$.  You will find that every fourth $k$ produces the correct modulo relations starting with $k=1$.  The $k=1$ solution finds the trivial $P_0 = H_0 = 1$, $k=5$ gives the solution suggested in the problem, and $k=9$ gives the desired value.

This direct calculation is obviously more involved in deriving, but requires little computation time.  For finding the $N^\mbox{th}$ time the three values coincide takes only $\mathcal O(N)$ time.

In [66]:
%%time

def p045(tot: int) -> int:
    u0 = 2
    v0 = 1
    uk = u0
    vk = v0
    for i in range(1, tot):
        for j in range(4):
            uk, vk = u0 * uk + 3 * v0 * vk, u0 * vk + uk * v0
    n = (uk + vk + 1) // 4
    return n * (2 * n - 1)

N = 3
ans = p045(N)

print(ans)

1533776805
CPU times: user 134 µs, sys: 6 µs, total: 140 µs
Wall time: 149 µs
